# 03 - Analisis de validacion

Hito de Semana 6: cruzar el mapa de angulo espectral contra la verdad de
terreno y medir. Es el paso que separa un mapa de similitud espectral de una
deteccion de caolinita, y el que contesta si el detector sirve.

El notebook no calcula nada por su cuenta: llama al paquete `mineralmap` y
comprueba que lo que devuelve cumple los contratos de
`docs/es/decisiones_tecnicas.md`, seccion 11. Todo lo que aqui se verifica esta
ademas cubierto por `tests/test_metrics.py`, que corre sin la escena.

**Adelanto del resultado: es negativo.** El ROC AUC vale 0,2403, por debajo del
0,5 del azar. No es un bug del notebook ni un error de signo; es lo que mide el
detector, y las celdas de abajo lo muestran desde tres angulos distintos.

Requiere que existan los dos GeoTIFF:

```bash
python scripts/run_pipeline.py --config configs/cerro_colorado_kaolinite.yaml
python scripts/construir_verdad_terreno.py
```

## 1. Setup

`autoreload` para que editar el paquete se refleje sin reiniciar el kernel.
El `chdir` deja el directorio de trabajo en la raiz del repositorio: las rutas
del proyecto son relativas a la raiz, y un notebook se puede lanzar tanto desde
ahi como desde `notebooks/`.

Los imports van todos juntos y antes de cualquier otra sentencia, para que
`ruff` no tenga que quejarse de E402 sobre este archivo.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio

from mineralmap.config import load_config
from mineralmap.pipeline import DETECTORS
from mineralmap.validation.geology import (
    CLASE_AMBIGUO,
    CLASE_NEGATIVO,
    CLASE_POSITIVO,
)
from mineralmap.validation.metrics import (
    confusion_matrix,
    curva_roc,
    evaluar_deteccion,
    f1_score,
    preparar_pares,
    roc_auc,
)
from mineralmap.visualization.validacion import (
    plot_curva_roc,
    plot_histograma_por_clase,
)

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

RAIZ = Path.cwd()
RUTA_PUNTAJE = "outputs/maps/kaolinite_sam_angle.tif"
RUTA_VERDAD = "outputs/maps/ground_truth.tif"
RUTA_CONFIG = "configs/cerro_colorado_kaolinite.yaml"

print("Directorio de trabajo:", RAIZ)

## 2. El umbral y la direccion salen del config, no de aqui

Dos datos que este notebook **no** puede inventarse.

El umbral es `algorithm.params.angle_threshold_rad`. La direccion del puntaje
sale de `Detector.higher_is_better`, que es su unica fuente de verdad. Para el
SAM vale `False`: el puntaje es el angulo espectral, y **menor angulo es mas
evidencia**.

Escribir cualquiera de los dos a mano aqui crearia una segunda fuente de verdad
que envejeceria en silencio. Con la direccion equivocada, el AUC sale como
`1 - AUC` sin que nada falle.

In [ ]:
config = load_config(RUTA_CONFIG)

NOMBRE_ALGORITMO = config.algorithm["name"]
UMBRAL = float(config.algorithm["params"]["angle_threshold_rad"])
HIGHER_IS_BETTER = DETECTORS[NOMBRE_ALGORITMO].higher_is_better

print(f"Algoritmo         : {NOMBRE_ALGORITMO}")
print(f"Umbral (config)   : {UMBRAL} rad")
print(f"higher_is_better  : {HIGHER_IS_BETTER}")
print()
print("Menor angulo = mas parecido a la firma. Por eso higher_is_better es False,")
print("y por eso `roc_auc` lo exige como argumento obligatorio.")

## 3. Cargar los dos rasteres y comprobar que estan alineados

Este es el chequeo que ninguna inspeccion de la salida puede reemplazar. Dos
rasteres de la misma forma pero distinta `transform` cubren trozos distintos del
desierto; emparejarlos por indice produce una matriz de confusion, un F1 y un
AUC perfectamente formados, calculados sobre pares de pixeles que **no
corresponden al mismo punto del terreno**. Los numeros salen dentro de rango y
la tabla se ve bien.

Por eso se comprueba y se corta, en vez de advertir.

In [ ]:
def abrir(ruta):
    with rasterio.open(ruta) as src:
        return src.read(1).astype(np.float64), {
            "crs": str(src.crs),
            "transform": tuple(round(v, 6) for v in src.transform[:6]),
            "forma": (src.height, src.width),
        }


puntaje, geo_puntaje = abrir(RUTA_PUNTAJE)
verdad, geo_verdad = abrir(RUTA_VERDAD)

for campo in ("crs", "transform", "forma"):
    assert geo_puntaje[campo] == geo_verdad[campo], (
        f"{campo} no coincide: {geo_puntaje[campo]} vs {geo_verdad[campo]}"
    )

print("Alineacion verificada")
print(f"  CRS       : {geo_puntaje['crs']}")
print(f"  Forma     : {geo_puntaje['forma']}")
print(f"  Transform : {geo_puntaje['transform']}")

## 4. La verdad de terreno tiene tres clases, y la tercera no se colapsa

`1` positivo, `0` negativo, `255` **ambiguo** (unidad no clasificada o pixel
fuera de todo poligono).

El `0` significa "la cartografia dice que no hay". El `255` significa "la
cartografia no dice". Colapsar el segundo en el primero convierte una ausencia
de informacion en evidencia negativa, y sobre este AOI eso regala millones de
verdaderos negativos gratis.

La celda de abajo cuantifica exactamente cuanto regalaria.

In [ ]:
valores, cuentas = np.unique(verdad[~np.isnan(verdad)], return_counts=True)
total = verdad.size

print(f"{'clase':<12} {'pixeles':>12} {'% del AOI':>12}")
print("-" * 38)
for valor, cuenta in zip(valores, cuentas):
    nombre = {
        CLASE_POSITIVO: "positivo",
        CLASE_NEGATIVO: "negativo",
        CLASE_AMBIGUO: "ambiguo",
    }.get(int(valor), f"desconocido({valor})")
    print(f"{nombre:<12} {cuenta:>12,} {100 * cuenta / total:>11.2f} %")
print("-" * 38)
print(f"{'TOTAL':<12} {total:>12,}")

n_positivo = int((verdad == CLASE_POSITIVO).sum())
n_negativo = int((verdad == CLASE_NEGATIVO).sum())
n_ambiguo = int((verdad == CLASE_AMBIGUO).sum())

exactitud_honesta = n_negativo / (n_positivo + n_negativo)
exactitud_inflada = (n_negativo + n_ambiguo) / total

print()
print("Exactitud de un detector que dice 'no' en todos los pixeles:")
print(f"  excluyendo los ambiguos (correcto)  : {exactitud_honesta:.4%}")
print(f"  colapsandolos a negativo (inflado)  : {exactitud_inflada:.4%}")

## 5. Un solo lugar decide que pixel entra al calculo

`preparar_pares` es el unico filtro del modulo. Un pixel entra si y solo si su
verdad de terreno es positivo o negativo, su puntaje no es `NaN` y la mascara
del `Scene` lo declara valido.

Que el filtro viva en una sola funcion es lo que impide que una metrica lo
aplique y otra no. Si cada una filtrara por su cuenta, la tabla podria mezclar
la matriz de confusion sobre 242.967 pixeles con el AUC sobre 3.993.936, en la
misma columna y sin que nada falle.

Los descartes se cuentan **con precedencia**, de modo que las cuatro causas mas
los evaluados sumen exactamente el total del AOI.

In [ ]:
pares = preparar_pares(puntaje, verdad)
d = pares.descartes

print(f"{'evaluados':<26} {d['evaluados']:>12,}")
print(f"{'  positivos':<26} {d['positivos']:>12,}")
print(f"{'  negativos':<26} {d['negativos']:>12,}")
print()
print("Descartados, por causa:")
for causa in ("ambiguo", "gt_nan", "score_nan", "mascara"):
    print(f"{'  ' + causa:<26} {d[causa]:>12,}")
print("-" * 39)
print(f"{'TOTAL del AOI':<26} {d['total']:>12,}")

suma = d["evaluados"] + d["ambiguo"] + d["gt_nan"] + d["score_nan"] + d["mascara"]
assert suma == d["total"], f"las causas no suman el total: {suma} != {d['total']}"
print()
print("Las causas suman exactamente el total. Sin solapamientos que explicar.")

## 6. La tabla de metricas

`evaluar_deteccion` recorre el camino completo: filtra, binariza con la
direccion declarada y calcula las seis metricas mas la calibracion.

La binarizacion respeta `higher_is_better`: para el SAM detecta
`angulo <= umbral`, igual que `Detector.detects`. Comparar a mano con `<=`
funcionaria hoy y fallaria en silencio con el primer detector que vaya al
reves.

In [ ]:
resultado = evaluar_deteccion(
    puntaje, verdad, UMBRAL, higher_is_better=HIGHER_IS_BETTER
)
m = resultado["matriz_confusion"]
met = resultado["metricas"]

print(f"Umbral: {UMBRAL} rad   higher_is_better: {HIGHER_IS_BETTER}")
print()
print("Matriz de confusion")
print(f"  verdaderos positivos (vp) {m['vp']:>12,}")
print(f"  falsos positivos     (fp) {m['fp']:>12,}")
print(f"  falsos negativos     (fn) {m['fn']:>12,}")
print(f"  verdaderos negativos (vn) {m['vn']:>12,}")
print()
print("Metricas")
for etiqueta, clave in (
    ("precision", "precision"),
    ("recall", "recall"),
    ("F1", "f1"),
    ("IoU", "iou"),
    ("kappa de Cohen", "kappa"),
    ("ROC AUC", "roc_auc"),
):
    print(f"  {etiqueta:<24} {met[clave]:>10.4f}")

### Como se lee esa tabla

**Los ceros son un solo hecho contado cinco veces.** Con el umbral del config no
hay ninguna deteccion dentro de las clases evaluables: las 131 detecciones del
AOI caen todas en pixeles ambiguos, y 130 de ellas sobre los botaderos de la
mina Cerro Colorado. Con `vp = fp = 0`, todo lo que se calcula desde la matriz
de confusion vale cero. No es un error: es lo que mide un detector que no
detecta.

**El numero que informa es el AUC, porque no depende del umbral.** Y esta por
debajo de 0,5, que es el valor del azar. Eso no significa que el detector no
separe: significa que **separa al reves**.

In [ ]:
y_true, y_score = pares.y_true, pares.y_score
angulos_positivo = y_score[y_true == 1]
angulos_negativo = y_score[y_true == 0]

print(f"{'':<14} {'positivo':>12} {'negativo':>12}")
print("-" * 40)
for etiqueta, f in (
    ("n", lambda a: a.size),
    ("minimo", lambda a: a.min()),
    ("mediana", np.median),
    ("maximo", lambda a: a.max()),
):
    p, n = f(angulos_positivo), f(angulos_negativo)
    if etiqueta == "n":
        print(f"{etiqueta:<14} {p:>12,} {n:>12,}")
    else:
        print(f"{etiqueta:<14} {p:>12.4f} {n:>12.4f}")

print()
print("La mediana del angulo es MAYOR dentro del positivo que dentro del negativo:")
print("las unidades que la cartografia declara compatibles con alteracion se")
print("parecen MENOS a la caolinita de laboratorio que las que declara no")
print("candidatas. Eso es lo que un AUC bajo 0,5 esta diciendo.")

## 7. La direccion del puntaje: el error mas caro del modulo

Un AUC calculado con la direccion invertida devuelve `1 - AUC`. Los dos numeros
son validos, ninguno lanza excepcion y los dos se pueden defender en una
presentacion: uno diria "el metodo funciona razonablemente bien" y el otro "el
metodo falla".

Por eso `roc_auc` exige `higher_is_better` **sin valor por defecto y solo por
palabra clave**. La celda de abajo muestra los dos numeros juntos, que es la
unica forma de que la trampa se vea.

In [ ]:
auc_declarado = roc_auc(y_true, y_score, higher_is_better=HIGHER_IS_BETTER)
auc_invertido = roc_auc(y_true, y_score, higher_is_better=not HIGHER_IS_BETTER)

print(f"AUC con la direccion declarada (menor = mejor) : {auc_declarado:.4f}")
print(f"AUC con la direccion invertida (mayor = mejor) : {auc_invertido:.4f}")
print(f"Suma                                           : "
      f"{auc_declarado + auc_invertido:.4f}")

assert abs(auc_declarado + auc_invertido - 1.0) < 1e-12

print()
print("El 0,7597 NO es un detector de alteracion que funciona al reves.")
print("Lo que separa en esa direccion no es caolinita: son diferencias")
print("litologicas generales entre roca intrusiva y relleno sedimentario, que")
print("el angulo contra CUALQUIER firma habria capturado igual.")

## 8. La figura: curva ROC e histograma por clase

Los dos paneles van juntos a proposito. Un AUC solo no distingue "las dos
distribuciones se superponen" de "estan separadas, pero al reves", y esas dos
lecturas llevan a decisiones opuestas. El segundo panel es el que lo dice.

Es la misma figura que produce `python scripts/evaluate.py ... --figura` en
`outputs/figures/roc_kaolinite_sam.png`.

In [ ]:
curva = curva_roc(y_true, y_score, higher_is_better=HIGHER_IS_BETTER)

fig, (ax_roc, ax_hist) = plt.subplots(1, 2, figsize=(15, 6.5))
plot_curva_roc(
    curva,
    title=f"Curva ROC --- SAM caolinita\n{y_true.size:,} pixeles evaluables",
    ax=ax_roc,
)
plot_histograma_por_clase(y_true, y_score, umbral=UMBRAL, ax=ax_hist)
fig.tight_layout()
plt.show()

Tres cosas que leer en esa figura:

1. **La curva pasa por debajo de la diagonal.** La diagonal es el azar; una
   curva por debajo es un detector que ordena al reves.
2. **Las dos distribuciones se solapan mucho, y la naranja esta a la derecha.**
   El positivo tiene angulos mas grandes que el negativo.
3. **El umbral del config (linea negra punteada, 0,1 rad) cae a la izquierda de
   las dos distribuciones.** No toca ninguna: por eso no hay ni una deteccion
   dentro de las clases evaluables.

## 9. La calibracion del umbral, y por que no rescata nada

`curva_roc` devuelve los dos umbrales operativos habituales, **en las unidades
originales del puntaje** (radianes de angulo, no el puntaje orientado), para que
se puedan comparar directamente con el `angle_threshold_rad` del config.

Se reportan los dos porque optimizan cosas distintas: Youden pesa igual los dos
tipos de error y F1 ignora los verdaderos negativos.

In [ ]:
print(f"{'criterio':<22} {'umbral (rad)':>14} {'valor':>10}")
print("-" * 48)
print(f"{'indice de Youden':<22} {curva.umbral_youden:>14.4f} "
      f"{curva.youden_j:>10.4f}")
print(f"{'F1 maximo':<22} {curva.umbral_f1:>14.4f} {curva.f1_maximo:>10.4f}")
print(f"{'config (heredado)':<22} {UMBRAL:>14.4f} {'-':>10}")

detectados_f1 = int((y_score <= curva.umbral_f1).sum())
tasa_base = float(y_true.mean())
f1_trivial = f1_score(y_true, np.ones_like(y_true))

print()
print(f"Con el umbral de F1 maximo se detectan {detectados_f1:,} de "
      f"{y_score.size:,} pixeles ({100 * detectados_f1 / y_score.size:.2f} %).")
print(f"La tasa base de positivos es {tasa_base:.4%}, y el F1 del clasificador")
print(f"que dice 'si' a TODO seria {f1_trivial:.4f}.")
print()
print("O sea: el 'umbral optimo por F1' es esencialmente el clasificador")
print("trivial. No es una calibracion, es la tasa base disfrazada de resultado.")
print("Y el de Youden alcanza J = "
      f"{curva.youden_j:.4f}, indistinguible de cero.")

## 10. Comprobaciones que hacen fallar el notebook si algo cambia

Igual que el notebook 01, este cierra con `assert`s. No son decorativos: si
alguien cambia el filtro de pixeles, la direccion del puntaje o el umbral del
config, alguna de estas lineas revienta y el notebook deja de correr entero.

Son las mismas cifras que publica `docs/es/decisiones_tecnicas.md`, seccion 11,
y que `scripts/verificar_cifras.py` coteja automaticamente contra los `.tif`.

In [ ]:
assert d["evaluados"] == 242_967, d["evaluados"]
assert d["positivos"] == 50_731, d["positivos"]
assert d["negativos"] == 192_236, d["negativos"]
assert d["ambiguo"] == 3_756_783, d["ambiguo"]

assert (m["vp"], m["fp"], m["fn"], m["vn"]) == (0, 0, 50_731, 192_236)

assert met["f1"] == 0.0
assert met["iou"] == 0.0
assert met["kappa"] == 0.0
assert abs(met["roc_auc"] - 0.2403) < 5e-5, met["roc_auc"]
assert met["roc_auc"] < 0.5, "el AUC dejo de estar bajo el azar; revisar"

# La matriz que se imprime arriba tiene que ser la misma que sale de binarizar
# a mano con la semantica del detector. Si esto falla, `evaluar_deteccion`
# binarizo en una direccion y el AUC se calculo en la otra.
y_pred = y_score <= UMBRAL if not HIGHER_IS_BETTER else y_score >= UMBRAL
assert confusion_matrix(y_true, y_pred)._asdict() == m

print("Todas las comprobaciones pasan.")
print()
print("Resultado de la validacion, sin adornos:")
print(f"  {d['evaluados']:,} pixeles evaluables")
print(f"  F1 = {met['f1']:.4f}   IoU = {met['iou']:.4f}   "
      f"kappa = {met['kappa']:.4f}")
print(f"  ROC AUC = {met['roc_auc']:.4f}  (el azar es 0,5)")
print()
print("El detector no encuentra alteracion argilica in situ en este AOI, y")
print("ningun umbral lo arregla. Las tres hipotesis --- que la arcilla expuesta")
print("este en los botaderos y no en la roca in situ, que el barniz del")
print("desierto enmascare la firma, o que 9 bandas a 20 m no basten --- estan")
print("en decisiones_tecnicas.md, seccion 11.1, sin elegir entre ellas.")